# STEP 4: 실제 OpenDART 재무 조회 Tool

이 노트북에서는 `src/dart/client.py`의 OpenDART 조회 함수를 LangChain Tool로 연결합니다.

`사용자 질문 → LLM의 Tool 요청 → OpenDART API 조회 → Tool 결과 → LLM 최종 답변`

이번 단계에서는 Tool 하나만 사용하며, 여러 Tool 선택·LangGraph·Agent는 만들지 않습니다.

## 0. 이번 Tool의 역할

Tool은 기업명과 사업연도를 받아 OpenDART에서 매출액, 영업이익, 당기순이익을 조회합니다. API 키는 Tool 인자가 아니라 notebook에서만 읽는 비밀값입니다. 따라서 LLM은 API 키를 보거나 생성하지 않습니다.

## 1. 프로젝트 경로와 API 키 준비

Jupyter를 프로젝트 루트 또는 `notebooks/`에서 열어도 `src/`와 `.env`를 찾도록 경로를 준비합니다. API 키 값은 출력하지 않습니다.

In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

current_dir = Path.cwd().resolve()
if current_dir.name == "learning":
    project_root = current_dir.parents[1]
elif current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
api_key = os.getenv("OPENDART_API_KEY")

if not api_key or api_key == "your_opendart_api_key":
    raise ValueError(".env의 OPENDART_API_KEY를 실제 API 키로 바꿔 주세요.")

print("프로젝트 경로와 OpenDART API 키를 준비했습니다.")

프로젝트 경로와 OpenDART API 키를 준비했습니다.


## 2. 재사용 가능한 OpenDART 함수와 LangChain 클래스 불러오기

`get_financial_summary()`는 LLM과 무관한 순수 Python 함수입니다. 기업명→`corp_code`→재무제표→주요 세 계정 요약을 담당합니다. notebook은 이 함수를 새로 구현하지 않고 import만 합니다.

In [2]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_ollama import ChatOllama

from src.dart.client import get_financial_summary

  from .autonotebook import tqdm as notebook_tqdm


## 3. OpenDART 조회 함수를 Tool로 감싸기

Tool의 인자는 `company_name`, `business_year` 두 개뿐입니다. `api_key`는 notebook 바깥으로 전달되지 않는 closure 변수이므로 Tool schema에 포함되지 않습니다. Tool 결과는 LLM이 읽기 쉬운 JSON 문자열로 반환합니다.

In [3]:
@tool
def get_financial_summary_from_opendart(company_name: str, business_year: int) -> str:
    """OpenDART에서 기업의 사업연도 기준 매출액, 영업이익, 당기순이익을 조회한다."""
    summary = get_financial_summary(
        corp_name=company_name,
        business_year=business_year,
        api_key=api_key,
    )
    return json.dumps(summary, ensure_ascii=False)

## 4. LLM에 전달할 Tool schema 확인

여기에는 기업명과 사업연도만 보이고 API 키는 없어야 합니다. 이 schema가 LLM이 Tool을 요청할 때 참고하는 계약입니다.

In [4]:
get_financial_summary_from_opendart.args_schema.model_json_schema()

{'description': 'OpenDART에서 기업의 사업연도 기준 매출액, 영업이익, 당기순이익을 조회한다.',
 'properties': {'company_name': {'title': 'Company Name', 'type': 'string'},
  'business_year': {'title': 'Business Year', 'type': 'integer'}},
 'required': ['company_name', 'business_year'],
 'title': 'get_financial_summary_from_opendart',
 'type': 'object'}

## 5. 로컬 Qwen과 Tool 연결

`bind_tools()`는 Tool schema를 모델에 전달할 뿐, OpenDART를 바로 호출하지는 않습니다. 현재 `ChatOllama`에서는 Tool 호출 강제 옵션을 지원하지 않으므로 system 메시지로 실제 재무 수치에는 Tool을 사용하도록 지시합니다.

In [5]:
llm = ChatOllama(
    model="qwen3.6:27b",
    base_url="http://localhost:11434",
    temperature=0,
)

llm_with_tools = llm.bind_tools([get_financial_summary_from_opendart])

## 6. 실제 재무 수치 질문하기

첫 모델 호출은 최종 답변을 만들기 위한 것이 아니라, Tool 이름과 인자가 담긴 `tool_calls` 요청을 받기 위한 것입니다.

In [6]:
question = "삼성전자의 2025년 매출액, 영업이익, 당기순이익을 알려줘."
system_message = SystemMessage(
    content=(
        "기업의 실제 재무 수치를 묻는 질문에는 반드시 제공된 OpenDART Tool을 사용한다. "
        "Tool 결과에 없는 수치는 추정하거나 생성하지 않는다. 값이 null이면 조회되지 않았다고 답한다."
    )
)
initial_messages = [system_message, HumanMessage(content=question)]

tool_request = llm_with_tools.invoke(initial_messages)
tool_request.tool_calls

[{'name': 'get_financial_summary_from_opendart',
  'args': {'company_name': '삼성전자', 'business_year': 2025},
  'id': 'd7a92b87-7133-4a0f-8194-b1451b869bc2',
  'type': 'tool_call'}]

## 7. 모델의 Tool 요청 확인

`args`에 `company_name: 삼성전자`, `business_year: 2025`가 들어 있는지 확인합니다. 아직 OpenDART 호출은 실행되지 않았습니다.

In [7]:
if not tool_request.tool_calls:
    raise RuntimeError("모델이 OpenDART Tool을 요청하지 않았습니다. tool_request 내용을 확인해 주세요.")

tool_call = tool_request.tool_calls[0]
if tool_call["name"] != get_financial_summary_from_opendart.name:
    raise ValueError(f"예상하지 않은 Tool 요청입니다: {tool_call['name']}")

print("Tool 인자:", tool_call["args"])

Tool 인자: {'company_name': '삼성전자', 'business_year': 2025}


## 8. Python이 실제 OpenDART Tool 실행

이 셀에서 처음으로 OpenDART API가 호출됩니다. `src/dart/client.py`가 기업명을 `corp_code`로 바꾸고 전체 재무제표에서 필요한 세 계정을 찾습니다.

In [8]:
tool_result = get_financial_summary_from_opendart.invoke(tool_call["args"])
tool_result

'{"기업명": "삼성전자", "사업연도": 2025, "매출액": "333605938000000", "영업이익": "43601051000000", "당기순이익": "45206805000000"}'

## 9. Tool 결과를 데이터로 다시 확인

Tool 결과는 LLM에 전달할 JSON 문자열입니다. 먼저 Python에서 다시 딕셔너리로 바꿔 실제 OpenDART 값인지 확인합니다.

In [9]:
financial_data = json.loads(tool_result)
financial_data

{'기업명': '삼성전자',
 '사업연도': 2025,
 '매출액': '333605938000000',
 '영업이익': '43601051000000',
 '당기순이익': '45206805000000'}

## 10. OpenDART 결과를 근거로 최종 답변 만들기

원래 질문, 모델의 Tool 요청, OpenDART Tool 결과를 순서대로 전달합니다. 모델은 Tool 결과에 없는 재무 수치를 만들어 내지 않도록 지시합니다.

In [10]:
messages = [
    *initial_messages,
    tool_request,
    ToolMessage(content=tool_result, tool_call_id=tool_call["id"]),
]

final_response = llm.invoke(messages)
print(final_response.content)

삼성전자의 2025년 재무 수치는 다음과 같습니다.

* **매출액**: 333조 6,059억 3,800만 원 (333,605,938,000,000 원)
* **영업이익**: 43조 6,010억 5,100만 원 (43,601,051,000,000 원)
* **당기순이익**: 45조 2,068억 500만 원 (45,206,805,000,000 원)


## 이번 단계에서 반드시 이해할 개념

1. LLM은 OpenDART API 키를 알지 못하며, 기업명과 사업연도만 Tool 인자로 요청합니다.
2. Tool 내부의 Python 코드가 실제 OpenDART API를 호출하고 원문 수치를 반환합니다.
3. LLM의 첫 응답은 답변이 아니라 Tool 실행 요청입니다.
4. 최종 답변은 `ToolMessage`로 전달된 실제 OpenDART 결과를 바탕으로 생성됩니다.
5. 지금은 Tool 하나를 사람이 직접 실행합니다. 여러 Tool 중 선택하거나 실패 시 재시도하는 workflow는 아직 만들지 않습니다.